# Assignment A2: Topic Modeling and Text ML

Covering material from Notebooks 5 and 6

In [ ]:
#Import the AG news dataset (same as hw01)
#Download them from here 
#!wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv

import pandas as pd
import nltk
df = pd.read_csv('train.csv')

df.columns = ["label", "title", "lead"]
label_map = {1:"world", 2:"sport", 3:"business", 4:"sci/tech"}
def replace_label(x):
	return label_map[x]
df["label"] = df["label"].apply(replace_label) 
df["text"] = df["title"] + " " + df["lead"]
df.head()


import spacy
dfs = df.sample(200)
nlp = spacy.load('en_core_web_md')

In [1]:
#Import the AG news dataset (same as hw01)
#Download them from here 
#!wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv

import pandas as pd
import nltk
df = pd.read_csv('train.csv')

df.columns = ["label", "title", "lead"]
label_map = {1:"world", 2:"sport", 3:"business", 4:"sci/tech"}
def replace_label(x):
	return label_map[x]
df["label"] = df["label"].apply(replace_label) 
df["text"] = df["title"] + " " + df["lead"]
df.head()


import spacy
dfs = df.sample(200)
nlp = spacy.load('en_core_web_md')

# A. Dimension Reduction

## PCA

In [4]:
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

pca = PCA(n_components=3, svd_solver='randomized')

##TODO reduce the vectorized data using PCA
# get spacy 300-dim vectors for each snippet, then fit-transform to 3 dims
vectors = np.array([nlp(text).vector for text in dfs["text"]])
vectors_pca = pca.fit_transform(vectors)

##TODO compute again cosine similarity with the reduced version for the first 200 snippets
cos_sim_pca = cosine_similarity(vectors_pca)
print(cos_sim_pca)

##TODO for the first snippet, show again its three most similar snippets
first_sim = cos_sim_pca[0]
top3_idx = np.argsort(first_sim)[::-1][1:4]  # skip index 0 (itself)
print("First snippet:")
print(dfs["text"].iloc[0])
print("\nThree most similar snippets:")
for idx in top3_idx:
    print(f"\nSimilarity: {first_sim[idx]:.4f}")
    print(dfs["text"].iloc[idx])


[[ 1.         -0.83949304 -0.77539206 ... -0.83700675 -0.68331397
   0.53882194]
 [-0.83949304  0.9999999   0.73635805 ...  0.80580837  0.48876414
  -0.878721  ]
 [-0.77539206  0.73635805  1.         ...  0.9927296   0.06907567
  -0.35373262]
 ...
 [-0.83700675  0.80580837  0.9927296  ...  0.99999994  0.17625517
  -0.44268107]
 [-0.68331397  0.48876414  0.06907567 ...  0.17625517  1.0000001
  -0.46418506]
 [ 0.53882194 -0.878721   -0.35373262 ... -0.44268107 -0.46418506
   1.        ]]
First snippet:
Shades of the Vioxx Case for Another Drug At a hearing Thursday on Capitol Hill, senators excoriated top federal drug regulators for failing to realize three years ago that Vioxx, a pain pill that Merck withdrew in September, was dangerous.

Three most similar snippets:

Similarity: 0.9842
Lord Hanson: a swashbuckling risk-taker For a man whose name hung above the door, James Hanson was remarkably unsentimental about the company he had built from scratch. As a champion of shareholder value

Compare the cosine similarity between docs before and after PCA reduction. Did the results change? 

## Topic Modeling with LDA

For this part you will need to use LDA Mallet. If you cannot have Mallet run, you can use the simple LDA algorithm 

In [6]:
from gensim.corpora import Dictionary
from gensim.models import LdaModel  # LdaMallet removed in gensim 4.x; using built-in LDA
from gensim.models.coherencemodel import CoherenceModel
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS

# tokenize and remove stopwords for LDA input
tokenized_text = [
    [token for token in simple_preprocess(doc) if token not in STOPWORDS]
    for doc in dfs["text"]
]

##TODO create a dictionary with the pre-processed tokenized text and filter it according to frequencies and keeping 1000 vocabularies
dictionary = Dictionary(tokenized_text)
dictionary.filter_extremes(no_below=2, no_above=0.5, keep_n=1000)

##TODO create the doc_term_matrix
doc_term_matrix = [dictionary.doc2bow(doc) for doc in tokenized_text]


In [7]:
##TODO train a LDA model with 5, 10 and 15 topics
##TODO compute the coherence score for each of these model and print the topics from the model with highest coherence score

coherence_scores = {}
lda_models = {}

for n_topics in [5, 10, 15]:
    model = LdaModel(corpus=doc_term_matrix, id2word=dictionary, num_topics=n_topics, random_state=42, passes=10)
    cm = CoherenceModel(model=model, texts=tokenized_text, dictionary=dictionary, coherence='c_v')
    coherence_scores[n_topics] = cm.get_coherence()
    lda_models[n_topics] = model
    print(f"n_topics={n_topics}  coherence={coherence_scores[n_topics]:.4f}")

best_n = max(coherence_scores, key=coherence_scores.get)
print(f"\nBest number of topics: {best_n} (coherence={coherence_scores[best_n]:.4f})")
print("\nTopics:")
for topic in lda_models[best_n].print_topics():
    print(topic)


n_topics=5  coherence=0.5090
n_topics=10  coherence=0.4858
n_topics=15  coherence=0.5034

Best number of topics: 5 (coherence=0.5090)

Topics:
(0, '0.014*"ap" + 0.011*"open" + 0.010*"reuters" + 0.008*"value" + 0.008*"wednesday" + 0.008*"technology" + 0.008*"st" + 0.008*"world" + 0.008*"stocks" + 0.008*"google"')
(1, '0.020*"reuters" + 0.016*"ap" + 0.016*"new" + 0.009*"iraq" + 0.008*"work" + 0.007*"state" + 0.007*"said" + 0.006*"lt" + 0.006*"gt" + 0.006*"support"')
(2, '0.017*"said" + 0.011*"year" + 0.010*"new" + 0.009*"microsoft" + 0.008*"world" + 0.008*"ap" + 0.008*"business" + 0.008*"space" + 0.007*"company" + 0.007*"london"')
(3, '0.014*"friday" + 0.014*"said" + 0.012*"oil" + 0.012*"reuters" + 0.012*"night" + 0.012*"gold" + 0.012*"world" + 0.010*"record" + 0.010*"wednesday" + 0.010*"new"')
(4, '0.016*"new" + 0.016*"peoplesoft" + 0.014*"south" + 0.013*"oracle" + 0.011*"business" + 0.009*"air" + 0.009*"world" + 0.009*"settle" + 0.007*"years" + 0.007*"european"')


In [8]:
import pyLDAvis
import pyLDAvis.gensim_models  # gensim_models replaces gensim in pyLDAvis >= 3.3

##TODO using LDAvis visualize the topics using the optimal number of topics
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_models[best_n], doc_term_matrix, dictionary)
vis


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
1     -0.098699 -0.084654       1        1  29.924480
2      0.066638 -0.018277       2        1  23.423196
0     -0.060598 -0.017110       3        1  19.265327
3     -0.039839  0.139724       4        1  14.710509
4      0.132498 -0.019682       5        1  12.676489, topic_info=           Term      Freq     Total Category  logprob  loglift
32   peoplesoft  5.000000  5.000000  Default  30.0000  30.0000
27     business  9.000000  9.000000  Default  29.0000  29.0000
31       oracle  5.000000  5.000000  Default  28.0000  28.0000
293        gold  6.000000  6.000000  Default  27.0000  27.0000
354         oil  9.000000  9.000000  Default  26.0000  26.0000
..          ...       ...       ...      ...      ...      ...
34        plans  1.592308  6.217420   Topic5  -5.3415   0.7033
440   president  1.588710  5.348777   Topic5  -5.3437   0.8515
410  executives  1.588645  4.549499   Topic5  -5.3438   1.0133
529         afp  1.588601  5.412305   Topic5  -5.3438   0.8396
89       second  1.588569  5.386154   Topic5  -5.3438   0.8444

[301 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
540       2  0.757533       afghan
540       4  0.252511       afghan
541       2  0.628843  afghanistan
541       4  0.209614  afghanistan
529       2  0.369528          afp
...     ...       ...          ...
9         5  0.322073        years
195       1  0.503091         york
195       3  0.125773         york
195       4  0.251545         york
195       5  0.125773         york

[459 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[2, 3, 1, 4, 5])

# B. Supervised Learning

## Load and Pre-process Text
We do sentiment analysis on the [Movie Review Data](https://www.cs.cornell.edu/people/pabo/movie-review-data/). If you would like to know more about the data, have a look at [the paper](https://www.cs.cornell.edu/home/llee/papers/pang-lee-stars.pdf) (but no need to do so).

In [11]:
# use curl (macOS default) instead of wget
!curl -L -o scale_data.tar.gz https://www.cs.cornell.edu/people/pabo/movie-review-data/scale_data.tar.gz
!curl -L -o scale_whole_review.tar.gz https://www.cs.cornell.edu/people/pabo/movie-review-data/scale_whole_review.tar.gz

!tar xf scale_data.tar.gz
!tar xf scale_whole_review.tar.gz


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 3935k  100 3935k    0     0  3636k      0  0:00:01  0:00:01 --:--:-- 3664k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 8645k  100 8645k    0     0  5832k      0  0:00:01  0:00:01 --:--:-- 5853k


First, we have to load the data for which we provide the function below. Note how we also preprocess the text using gensim's simple_preprocess() function and how we already split the data into a train and test split.

In [12]:
import os
from gensim.utils import simple_preprocess
def load_data():
    examples, labels = [], []
    authors = os.listdir("scale_whole_review")
    for author in authors:
        path = os.listdir(os.path.join("scale_whole_review", author, "txt.parag"))
        fn_ids = os.path.join("scaledata", author, "id." + author)
        fn_ratings = os.path.join("scaledata", author, "rating." + author)
        with open(fn_ids) as ids, open(fn_ratings) as ratings:
            for idx, rating in zip(ids, ratings):
                labels.append(float(rating.strip()))
                filename_text = os.path.join("scale_whole_review", author, "txt.parag", idx.strip() + ".txt")
                with open(filename_text, encoding='latin-1') as f:
                    examples.append(" ".join(simple_preprocess(f.read())))
    return examples, labels
                  
X,y  = load_data()
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
print ("text:", X_train[0], "\nlabel:", y_train[0])

text: bloody child the director writer cinematographer nina menkes screenwriter tinka menkes editors nina and tina menkes cast tinka menkes captain sherry sibley murdered wife robert mueller murderer russ little sergeant jack hara enlisted man runtime mirage reviewed by dennis schwartz an amazingly strange film confusing and not thoroughly enjoyable but film found more interesting than thought possible at first viewing this experimental film in minimalist story telling film consisting of disturbing visualizations and almost no dialogue had concept that was greater than how the film turned out it felt at times like was watching paint dry on the wall but the reward for sitting through those excruciatingly redundant scenes was in seeing something different something that cast spell of sorcery over terrible incident as believe the film in its unique and sometimes shrill voice does justice in commenting on the violence in american society especially against women the film uses its impressio

## Vectorize the data

In [13]:
# train a TF_IDF Vectorizer on X_train and vectorize X_train and X_test
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(min_df=0.01, # at min 1% of docs
                        max_df=.5,  
                        stop_words='english',
                        ngram_range=(1,2))

##TODO train vectorizer
vec.fit(X_train)

##TODO transform X_train to TF-IDF values
X_train_tfidf = vec.transform(X_train)
##TODO transform X_test to TF-IDF values
X_test_tfidf = vec.transform(X_test)


In [14]:
##TODO scale both training and test data with the standard scaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler(with_mean=False)

X_train_tfidf = scaler.fit_transform(X_train_tfidf)
X_test_tfidf = scaler.transform(X_test_tfidf)


## ElasticNet

In [15]:
##TODO train an elastic net on the transformed output of the scaler
from sklearn.linear_model import ElasticNet

en = ElasticNet(alpha=0.01)

##TODO train the ElasticNet
en.fit(X_train_tfidf, y_train)

##TODO predict the testset
y_pred = en.predict(X_test_tfidf)

from sklearn.metrics import r2_score, mean_squared_error
##TODO print mean squared error and r2 score on the test set
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"R2:  {r2_score(y_test, y_pred):.4f}")


MSE: 0.0165
R2:  0.4977


## Logistic Regression

Next, we train an OLS model doing binary prediction on these movie reviews. Two get two bins, we transform the continuous ratings into two classes, where one class contains all the negative ratings (value < 0.5), the other class all the positive ratings (value > 0.5)

In [16]:
y_train = [1 if i >= 0.5 else 0 for i in y_train]
y_test = [1 if i >= 0.5 else 0 for i in y_test]


In [17]:
##TODO train logistic regression on X_train
from sklearn.linear_model import LogisticRegression
import numpy as np
logistic_regression = LogisticRegression()

##TODO train a logistic regression
logistic_regression.fit(X_train_tfidf, y_train)

##TODO predict the testset
y_pred_proba = logistic_regression.predict_proba(X_test_tfidf)[:, 1]

##since we have continuous output, we need to post-process our labels into two classes. We choose a threshold of 0.5 
def map_predictions(predicted):
    predicted = [1 if i > 0.5 else 0 for i in predicted]
    return predicted

y_pred_binary = map_predictions(y_pred_proba)

##TODO print the accuracy of our classifier on the testset
from sklearn.metrics import accuracy_score
print(f"Accuracy: {accuracy_score(y_test, y_pred_binary):.4f}")

## TODO print the 10 most informative words of the regression (the 10 words having the highest coefficients)
feature_names = vec.get_feature_names_out()
top10_idx = np.argsort(logistic_regression.coef_[0])[-10:][::-1]
print("\n10 most informative words:")
for idx in top10_idx:
    print(f"  {feature_names[idx]:<30} coef={logistic_regression.coef_[0][idx]:.4f}")


/opt/anaconda3/envs/tad_courses/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(


Accuracy: 0.8111

10 most informative words:
  great                          coef=0.2331
  surprisingly                   coef=0.2182
  best                           coef=0.2119
  effective                      coef=0.2049
  fascinating                    coef=0.1984
  success                        coef=0.1942
  punches                        coef=0.1941
  elements                       coef=0.1906
  brilliant                      coef=0.1902
  fine                           coef=0.1901


## XGBoost

Lastly, we train an XGBoost classifier to do topic prediction on the AG news dataset, which is a multi-class prediction problem (4 classes). We again have to vectorize the data, train the classifier, predict the testset and output an evaluation metric (we go for accuracy).

In [ ]:
!pip install xgboost

In [18]:

# vectorize the data
from sklearn.feature_extraction.text import TfidfVectorizer

# only consider 10% of the data
dfs = df.sample(frac=0.1)

# split into train and test
X_train, X_test, y_train, y_test = train_test_split(dfs["text"], dfs["label"], test_size=0.33, random_state=42)

vec = TfidfVectorizer(min_df=5, # at min 1% of docs
                        max_df=.5,  
                        stop_words='english',
                        max_features=2000,
                        ngram_range=(1,2))

# transform into TF-IDF values
X_train_tfidf = vec.fit_transform(X_train).todense()
X_test_tfidf = vec.transform(X_test).todense()


XGBoost provides an interface to SKLearn classifiers, e.g. they implement the same train and predict methods as an SKLearn classifier would. If you are interested in a more detailed overview, have a look at the [official documentation](https://xgboost.readthedocs.io/en/latest/python/index.html).

In [19]:
param_dist = {'objective':'multi:softmax', 'num_class': 5, 'n_estimators':25}
# note how we only have 4 labels, but we need to pass "num_class": 5
# if we pass "num_class": 4, we get the error "label must be in [0, num_class)."
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

clf = xgb.XGBModel(**param_dist)

# XGBoost requires integer labels; encode string labels to 0-3
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

##TODO train the XGBModel
clf.fit(X_train_tfidf, y_train_enc)

##TODO predict the testset
y_pred = clf.predict(X_test_tfidf).astype(int)

##TODO evaluate the predictions using accuracy as a metric
print(f"Accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")


Accuracy: 0.8066
